In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)




df = pd.read_csv('cardio_cleaned.csv')
df = df.drop_duplicates()                       # شيل الصفوف المكررة
df = df.drop(columns=['age'])                   # نفس معلومة age_years
df.loc[df['height'] < 140, 'height'] = np.nan   # الطول المستحيل -> NaN


X = df.drop(columns=['cardio'])
y = df['cardio']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)




log_reg = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(C=1.0, max_iter=1000)),
])
log_reg.fit(X_train, y_train)




weights = pd.Series(log_reg.named_steps['model'].coef_[0], index=X.columns)
print('Weights:\n', weights.sort_values(ascending=False).round(3), '\n')
print('Bias:', log_reg.named_steps['model'].intercept_.round(3), '\n')




def evaluate(model, name):
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, proba),
    }

results = [evaluate(log_reg, 'Logistic Regression (baseline)')]
print('Confusion Matrix (baseline):\n', confusion_matrix(y_test, log_reg.predict(X_test)), '\n')




print('Baseline train acc:', round(log_reg.score(X_train, y_train), 4),
      '| test acc:', round(log_reg.score(X_test, y_test), 4))
cv = cross_val_score(log_reg, X_train, y_train, cv=5, scoring='accuracy')
print('Baseline 5-fold CV acc:', cv.round(4), 'mean =', cv.mean().round(4), '\n')





tree = Pipeline([('imputer', SimpleImputer(strategy='median')),
                 ('model', DecisionTreeClassifier(random_state=42))])
tree.fit(X_train, y_train)
print('Decision Tree (no limit) train acc:', round(tree.score(X_train, y_train), 4),
      '| test acc:', round(tree.score(X_test, y_test), 4), '  <-- Overfitting!')
results.append(evaluate(tree, 'Decision Tree (no tuning)'))

rf = Pipeline([('imputer', SimpleImputer(strategy='median')),
               ('model', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))])
rf.fit(X_train, y_train)
results.append(evaluate(rf, 'Random Forest (default)'))





grid_lr = GridSearchCV(log_reg, {'model__C': [0.01, 0.1, 1, 10]}, cv=5, scoring='accuracy')
grid_lr.fit(X_train, y_train)
print('\nBest C for LogReg:', grid_lr.best_params_, 'CV acc =', round(grid_lr.best_score_, 4))
results.append(evaluate(grid_lr.best_estimator_, 'Logistic Regression (tuned)'))

grid_rf = GridSearchCV(rf, {'model__max_depth': [5, 8, 12],
                            'model__min_samples_leaf': [5, 20]},
                       cv=3, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train, y_train)
print('Best RF params:', grid_rf.best_params_, 'CV acc =', round(grid_rf.best_score_, 4))
results.append(evaluate(grid_rf.best_estimator_, 'Random Forest (tuned)'))





table = pd.DataFrame(results).set_index('Model').round(4)
print('\n', table.to_string())






best_rf = grid_rf.best_estimator_.named_steps['model']
print('\nRF importance:\n', pd.Series(best_rf.feature_importances_, index=X.columns)
      .sort_values(ascending=False).round(3))






new_patient = pd.DataFrame([{
    'gender': 2, 'height': 175, 'weight': 85, 'ap_hi': 150, 'ap_lo': 95,
    'cholesterol': 2, 'gluc': 1, 'smoke': 0, 'alco': 0, 'active': 1, 'age_years': 55}])
best_lr = grid_lr.best_estimator_
print('\nNew patient -> prediction:', best_lr.predict(new_patient)[0],
      '| probability:', best_lr.predict_proba(new_patient)[0, 1].round(3))

Weights:
 ap_hi          0.941
cholesterol    0.353
age_years      0.344
weight         0.157
ap_lo          0.107
gender        -0.009
height        -0.037
alco          -0.043
smoke         -0.044
gluc          -0.076
active        -0.087
dtype: float64 

Bias: [0.03] 

Confusion Matrix (baseline):
 [[5426 1510]
 [2322 4471]] 

Baseline train acc: 0.7291 | test acc: 0.7209
Baseline 5-fold CV acc: [0.7305 0.7305 0.7287 0.7284 0.7263] mean = 0.7289 

Decision Tree (no limit) train acc: 0.9954 | test acc: 0.6286   <-- Overfitting!

Best C for LogReg: {'model__C': 1} CV acc = 0.7289
Best RF params: {'model__max_depth': 12, 'model__min_samples_leaf': 20} CV acc = 0.7361

                                 Accuracy  Precision  Recall      F1  ROC-AUC
Model                                                                       
Logistic Regression (baseline)    0.7209     0.7475  0.6582  0.7000   0.7877
Decision Tree (no tuning)         0.6286     0.6244  0.6258  0.6251   0.6285
Random Forest 

In [2]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.4 MB 988.7 kB/s eta 0:00:08
   --- ------------------------------------ 0.8/8.4 MB 1.3 MB/s eta 0:00:07
   ----- ---------------------------------- 1.0/8.4 MB 1.2 MB/s eta 0:00:07
   ------ --------------------------------- 1.3/8.4 MB 1.2 MB/s eta 0:00:07
   ------- -------------------------------- 1.6/8.4 MB 1.1 MB/s eta 0:00:06
   ------- -------------------------------- 1.6/8.4 MB 1.1 MB/s eta 0:00:06
   ---------- ----------------------------- 2.1/8.4 MB 1.2 MB/s eta 0:00:06
   ---------- ----------------------------- 2.1/8.4 MB 1.2 MB/s eta 0:00:06
   ------------ --------------------------- 2.6/8.4 MB 1.2 MB/s eta 0:00:05
   ------------- -------------------------- 2.9/8.4 MB 1.2 MB/s eta 0:00:05
   --------------- ------------


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
